# Parallactic angle for LSST/Rubin

Ce notebook calcule et visualise l'**angle parallactique** $q$ en fonction de l'**angle horaire** $H$ pour :

1. Les champs profonds (DDF) de LSST (COSMOS, ECDFS, EDFS, M49).
2. Une grille de déclinaisons $\delta$ allant de $-90°$ à $+10°$ par pas de $10°$, avec une
   palette de couleurs divergente centrée sur la **latitude de l'observatoire**
   ($\phi_{\rm Rubin} \simeq -30.24°$) : les courbes correspondant à $\delta \approx \phi$ sont
   les plus « plates » (variation d'angle parallactique minimale au transit).

## Rappel de la formule

L'angle parallactique $q$ est défini par :

$$\tan q = \frac{\sin H}{\tan\phi\,\cos\delta - \sin\delta\,\cos H}$$

avec :
- $H$ : angle horaire de l'objet ($H = \mathrm{LST} - \alpha$)
- $\delta$ : déclinaison de la source
- $\phi$ : latitude géographique de l'observatoire

En pratique on utilise `numpy.arctan2(sin H, dénominateur)` pour obtenir l'angle dans $[-180°, +180°]$.

## Pertinence pour les dipôles LSST

- L'**amplitude** du dipôle de soustraction PSF est $\propto \sin z$ (angle zénithal).
- L'**orientation** du dipôle est donnée par $q(H)$.
- Quand $\delta \approx \phi$ (objet culminant au zénith), $\sin z$ reste faible autour du transit
  et $q$ varie peu : c'est le régime le plus favorable pour la calibration différentielle.

---
## 1. Imports et configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.dates as mdates
from matplotlib.colorbar import ColorbarBase
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u
from astropy.timeseries import TimeSeries
from astropy.coordinates import get_sun

from astroplan import Observer
from astroplan import FixedTarget
from astroplan.plots import plot_airmass, plot_parallactic, plot_altitude, plot_sky
from astroplan import is_observable

from pytz import timezone

warnings.filterwarnings("ignore")
print(f"pandas   version : {pd.__version__}")
print(f"numpy    version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

---
## 2. Répertoires de sortie

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
NB_TAG = "TOOLS_06_DDF-PARAANGLE"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Figures: {os.path.abspath(DIR_FIGS)}")

In [ ]:
def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")

---
## 3. Rappels théoriques

### 3.1 Cosinus de l'angle zénithal

On note :
- $H$ : angle horaire
- $\delta$ : déclinaison de la source
- $\phi$ : latitude de l'observatoire
- $z$ : angle zénithal

$$\cos z = \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H$$

### 3.2 Sinus de l'angle zénithal

$$\sin z = \sqrt{1 - \left(\sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H\right)^2}$$

En passant par l'altitude $h = \pi/2 - z$ :

$$\sin h = \sin\phi\sin\delta + \cos\phi\cos\delta\cos H \quad\Rightarrow\quad \sin z = \cos h$$

### 3.3 Interprétation physique pour les dipôles

| Quantité | Rôle dans le dipôle de soustraction |
|---|---|
| $\sin z(H)$ | **amplitude** du résidu dipôlaire |
| $q(H)$ | **orientation** (angle parallactique) du dipôle |

---
## 4. Fonctions utilitaires

In [ ]:
def zenith_tangent_vector(ra_deg, dec_deg, obstime, location):
    """
    Projection du zénith dans le plan tangent à la source (méthode vectorielle).

    v = z - (z·s) s,  avec s = direction source, z = direction zénith (ICRS)

    Parameters
    ----------
    ra_deg, dec_deg : coordonnées équatoriales J2000 de la cible
    obstime         : Time astropy
    location        : EarthLocation astropy

    Returns
    -------
    v : vecteur 3D unitaire dans le plan tangent
    """
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)
    zenith_altaz = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=obstime, location=location))
    zenith_icrs = zenith_altaz.transform_to("icrs")

    s = sky.cartesian.xyz.value
    z = zenith_icrs.cartesian.xyz.value

    v = z - np.dot(z, s) * s
    v /= np.linalg.norm(v)
    return v

In [ ]:
def zenith_tangent_vector_fromHA(HA_deg, coords, location):
    """
    Projection vectorielle du zénith dans le plan tangent, paramétrée par HA.

    Parameters
    ----------
    HA_deg   : Angle astropy (tableau), angle horaire en degrés
    coords   : SkyCoord de la cible
    location : EarthLocation

    Returns
    -------
    v_unit : (3, N) vecteurs unitaires tangents
    norm   : (N,)   norme avant normalisation (~sin z)
    """
    lat_deg = location.lat.to(u.deg).value
    dec_deg = coords.dec.to(u.deg).value
    ra_deg = coords.ra.to(u.deg).value

    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    s = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])

    HA_val = HA_deg.to(u.deg).value
    lst = np.deg2rad(HA_val + ra_deg)
    lat = np.deg2rad(lat_deg)

    z = np.array([np.cos(lat) * np.cos(lst), np.cos(lat) * np.sin(lst), np.sin(lat) * np.ones_like(lst)])

    proj = np.sum(z * s[:, None], axis=0)
    v = z - proj * s[:, None]
    norm = np.linalg.norm(v, axis=0)

    v_unit = np.zeros_like(v)
    mask = norm > 0
    v_unit[:, mask] = v[:, mask] / norm[mask]

    return v_unit, norm

In [ ]:
def sinz_vs_HA(HA_deg, coords, location):
    """
    Sinus de l'angle zénithal en fonction de l'angle horaire.

    sin z = sqrt(1 - (sin phi sin delta + cos phi cos delta cos H)^2)

    Parameters
    ----------
    HA_deg   : Angle astropy, angle horaire
    coords   : SkyCoord cible
    location : EarthLocation

    Returns
    -------
    sinz : ndarray
    """
    lat_deg = location.lat.to(u.deg).value
    dec_deg = coords.dec.to(u.deg).value
    HA = np.deg2rad(HA_deg.to(u.deg).value)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)

In [ ]:
def calculate_parallactic_angle(coords, times, location):
    """
    Angle parallactique à partir d'une grille de temps astropy.

    q = arctan2(sin H,  tan phi cos delta - sin delta cos H)

    Parameters
    ----------
    coords   : SkyCoord cible
    times    : Time astropy (tableau)
    location : EarthLocation

    Returns
    -------
    q_deg : ndarray, degrés
    """
    lst = times.sidereal_time("apparent", longitude=location.lon)
    H = (lst - coords.ra).to(u.rad).value
    phi = location.lat.to(u.rad).value
    dec = coords.dec.to(u.rad).value

    num = np.sin(H)
    den = np.tan(phi) * np.cos(dec) - np.sin(dec) * np.cos(H)
    return np.degrees(np.arctan2(num, den))

In [ ]:
def calculate_parallactic_angle_fromHA(ha, coords, location):
    """
    Angle parallactique à partir de l'angle horaire HA.

    q = arctan2(sin H,  tan phi cos delta - sin delta cos H)

    Parameters
    ----------
    ha       : Angle astropy, angle horaire
    coords   : SkyCoord cible
    location : EarthLocation

    Returns
    -------
    q_deg : ndarray, degrés
    """
    ha_rad = ha.to(u.rad).value
    phi = location.lat.to(u.rad).value
    dec = coords.dec.to(u.rad).value

    num = np.sin(ha_rad)
    den = np.tan(phi) * np.cos(dec) - np.sin(dec) * np.cos(ha_rad)
    return np.degrees(np.arctan2(num, den))

---
## 5. Initialisations : champs cibles et observatoire

### 5.1 Champs profonds LSST (DDF)

In [ ]:
# ── LSST Deep Drilling Fields ──────────────────────────────────────────────────
# Format : { nom : (RA_deg, Dec_deg) }
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ECDFS": (53.1250, -27.800),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# Couleurs pour le polar plot / autres visualisations
DEEP_FIELDS_COLORSTYLE = {
    "COSMOS": {"color": "r"},
    "ELAIS-S1": {"color": "k"},
    "ECDFS": {"color": "grey"},
    "EDFS-a": {"color": "b"},
    "EDFS-b": {"color": "g"},
    "EDFS": {"color": "magenta"},
    "M49": {"color": "purple"},
}

DEEP_FIELDS_LINESTYLE = {
    "COSMOS": {"ls": "-"},
    "ELAIS-S1": {"ls": "."},
    "ECDFS": {"ls": ":"},
    "EDFS-a": {"ls": ":"},
    "EDFS-b": {"ls": ":"},
    "EDFS": {"ls": "-."},
    "M49": {"ls": "--"},
}

### 5.2 Site de l'observatoire Rubin/LSST (Cerro Pachón)

In [ ]:
# ── Rubin/LSST observatory location (Cerro Pachón) ────────────────────────────
RUBIN_LAT_DEG = -30.244728  # degrés Nord  ← latitude critique pour les dipôles
RUBIN_LON_DEG = -70.749417  # degrés Est   (Ouest négatif)
RUBIN_HEIGHT_M = 2647.0  # mètres

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Rubin/LSST : lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

### 5.3 Observateur astroplan (contrôle)

In [ ]:
# Observateur astroplan (utilisé pour plot_parallactic, plot_airmass, etc.)
observer = Observer.at_site("lsst", timezone="UTC")
observer

---
## 6. Angle parallactique $q(H)$ pour les DDF

Chaque courbe correspond à l'un des quatre champs profonds principaux.
La déclinaison de chaque champ est rappelée en légende.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")

# Grille d'angles horaires : −180° à +180°
HA = np.arange(-180.0, 181.0) * u.deg

for key, value in DEEP_FIELDS.items():
    ra, dec = value
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = f"{key}  ($\\delta = {dec:.1f}°$)"

    q = calculate_parallactic_angle_fromHA(HA, coords, RUBIN_LOCATION)
    ax.plot(HA, q, label=label, lw=2)

# Axes
ax.set_xlabel("Angle horaire (degrés)")
ax.set_ylabel("Angle parallactique $q$ (degrés)")

# Axe secondaire en heures
secax = ax.secondary_xaxis("top", functions=(lambda x: x / 15.0, lambda x: x * 15.0))
secax.set_xlabel("Angle horaire (heures)")

ax.axvline(0, color="k", ls=":", lw=0.8, alpha=0.5)  # transit
ax.axhline(0, color="k", ls=":", lw=0.8, alpha=0.5)
ax.legend(shadow=True, loc="upper left")
ax.set_title("Angle parallactique – LSST Deep Drilling Fields")

savefig("LSST-DDF-ParallacticAngle")
plt.show()

---
## 7. Angle parallactique $q(H)$ — balayage en déclinaison

On fait varier $\delta$ de $-90°$ à $+10°$ par pas de $10°$.  
La palette de couleurs **divergente** (`RdBu_r`) est centrée sur la **latitude de Rubin**
($\phi \simeq -30.24°$) : les courbes bleues sont pour $\delta < \phi$, rouges pour $\delta > \phi$,
et la courbe **la plus proche de blanc** correspond à $\delta \approx \phi$.

> **Hypothèse à vérifier** : la courbe $q(H)$ est-elle la plus « plate »
> (i.e. variation minimale entre $H=-6h$ et $H=+6h$) quand $\delta \approx \phi$ ?
> Spoiler : non — la platitude dépend aussi du comportement de $\tan\phi\cos\delta - \sin\delta\cos H$
> autour du transit.

In [ ]:
# ── Paramètres du balayage ─────────────────────────────────────────────────────
DEC_MIN = -90.0  # degré
DEC_MAX = 10.0  # degré
DEC_STEP = 10.0  # pas

dec_values = np.arange(DEC_MIN, DEC_MAX + DEC_STEP, DEC_STEP)  # −90 … +10

# ── Colormap divergente centrée sur phi_Rubin ──────────────────────────────────
# On normalise de sorte que RUBIN_LAT_DEG tombe au centre (0.5) de la colormap.
# TwoSlopeNorm(vcenter=phi, vmin=dec_min, vmax=dec_max)
norm_cmap = mcolors.TwoSlopeNorm(
    vcenter=RUBIN_LAT_DEG,
    vmin=DEC_MIN,
    vmax=DEC_MAX,
)
cmap = cm.RdBu_r  # rouge = haute dec, bleu = basse dec

# ── Grille HA ─────────────────────────────────────────────────────────────────
HA = np.arange(-180.0, 181.0) * u.deg
HA_hours = HA.value / 15.0  # pour l'axe secondaire

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")

for dec in dec_values:
    # RA arbitraire (q ne dépend que de HA, pas de RA)
    coords = SkyCoord(0.0 * u.deg, dec * u.deg, frame="icrs")
    color = cmap(norm_cmap(dec))

    # Mise en évidence de la courbe la plus proche de phi_Rubin
    is_lat_curve = np.isclose(dec, np.round(RUBIN_LAT_DEG / DEC_STEP) * DEC_STEP, atol=DEC_STEP / 2)
    lw = 3.0 if is_lat_curve else 1.5
    alpha = 1.0 if is_lat_curve else 0.75
    zord = 5 if is_lat_curve else 2

    q = calculate_parallactic_angle_fromHA(HA, coords, RUBIN_LOCATION)
    label = f"$\\delta={dec:+.0f}°$" + (r" ← $\approx\phi_{\rm Rubin}$" if is_lat_curve else "")
    ax.plot(HA.value, q, color=color, lw=lw, alpha=alpha, zorder=zord, label=label if is_lat_curve else None)

# Superposition des DDF pour référence
for key, value in DEEP_FIELDS.items():
    ra, dec = value
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    q = calculate_parallactic_angle_fromHA(HA, coords, RUBIN_LOCATION)
    linestyle = DEEP_FIELDS_LINESTYLE[key]["ls"]
    ax.plot(
        HA.value,
        q,
        ls=linestyle,
        lw=2,
        color="k",
        alpha=0.55,
        zorder=10,
        label=f"{key} ($\\delta={dec:.1f}°$)",
    )

# ── Repères ──────────────────────────────────────────────────────────────────
ax.axvline(0, color="k", ls=":", lw=0.8, alpha=0.4)  # transit
ax.axhline(0, color="k", ls=":", lw=0.8, alpha=0.4)

# ── Axes ─────────────────────────────────────────────────────────────────────
ax.set_xlabel("Hour Angle  $H$ (degres)", fontsize=12)
ax.set_ylabel("Paralactic Angle $q$ (degres)", fontsize=12)

secax = ax.secondary_xaxis("top", functions=(lambda x: x / 15.0, lambda x: x * 15.0))
secax.set_xlabel("Angle horaire $H$ (heures)", fontsize=11)

# ── Colorbar ─────────────────────────────────────────────────────────────────
# On crée un ScalarMappable pour la colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label("Declinaison $\\delta$ (degres)", fontsize=11)

# Marquer la latitude Rubin sur la colorbar
cbar.ax.axhline(RUBIN_LAT_DEG, color="k", lw=1.5, ls="--")
cbar.ax.text(
    1.05,
    norm_cmap(RUBIN_LAT_DEG),
    f"$\\phi_{{\\rm Rubin}}={RUBIN_LAT_DEG:.1f}°$",
    transform=cbar.ax.transAxes,
    va="center",
    fontsize=9,
    color="k",
)

# ── Légende (DDF + courbe phi) ────────────────────────────────────────────────
ax.legend(fontsize=9, loc="lower right", ncol=1, framealpha=0.85)

ax.set_title(
    f"Parallactic Angle $q(H)$ — declinaison scan\n"
    f"Rubin/LSST, $\\phi = {RUBIN_LAT_DEG:.2f}°$ (colormap center)",
    fontsize=12,
)

savefig("LSST-ParallacticAngle-DecSweep")
plt.show()

### 7.1 Quantification de la « platitude » de $q(H)$

Pour chaque déclinaison on calcule la **variation totale** de $q$ sur la fenêtre
$H \in [-90°, +90°]$ (soit $\pm 6$ heures), qui correspond à l'intervalle d'observabilité typique.

In [ ]:
# Fenêtre d'observabilité raisonnable : |H| < 90° (= 6 h)
HA_range = np.linspace(-90.0, 90.0, 361) * u.deg
obs_window = np.abs(HA_range.value) <= 90.0

dq_list = []
for dec in dec_values:
    coords = SkyCoord(0.0 * u.deg, dec * u.deg, frame="icrs")
    q = calculate_parallactic_angle_fromHA(HA_range, coords, RUBIN_LOCATION)
    # Variation totale (max − min) sur la fenêtre
    dq_list.append(np.ptp(q[obs_window]))

dq_arr = np.array(dq_list)
idx_min = np.argmin(dq_arr)

print("Déclinaison avec la variation minimale de q sur |H| < 90° :")
print(f"  δ = {dec_values[idx_min]:.0f}°   Δq = {dq_arr[idx_min]:.1f}°")
print(f"  (latitude Rubin : φ = {RUBIN_LAT_DEG:.2f}°)")

# ── Bar plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")

colors = [cmap(norm_cmap(d)) for d in dec_values]
bars = ax.bar(dec_values, dq_arr, width=8.0, color=colors, edgecolor="k", linewidth=0.5)

# Repère latitude Rubin
ax.axvline(RUBIN_LAT_DEG, color="k", ls="--", lw=1.5, label=f"$\\phi_{{\\rm Rubin}} = {RUBIN_LAT_DEG:.1f}°$")
# Repère déclinaison à variation minimale
ax.axvline(
    dec_values[idx_min],
    color="firebrick",
    ls="-.",
    lw=1.5,
    label=f"$\\delta_{{\\min\\Delta q}} = {dec_values[idx_min]:.0f}°$",
)

ax.set_xlabel("Déclinaison $\\delta$ (degrés)", fontsize=11)
ax.set_ylabel("$\\Delta q_{|H|<90°}$ (degrés)", fontsize=11)
ax.set_title("Variation totale de l'angle parallactique sur $|H| < 90°$ = 6 h", fontsize=12)
ax.legend(fontsize=10)

savefig("LSST-ParallacticAngle-Flatness")
plt.show()

---
## 8. Sinus de l'angle zénithal $\sin z(H)$ pour les DDF

$\sin z$ module l'**amplitude** du dipôle de soustraction.  
Un champ qui culmine près du zénith ($\delta \approx \phi$) présente un $\sin z$ minimal au transit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")

HA = np.arange(-180.0, 181.0) * u.deg

# ── Méthode vectorielle ────────────────────────────────────────────────────────
ax = axes[0]
for key, (ra, dec) in DEEP_FIELDS.items():
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    _, sinz = zenith_tangent_vector_fromHA(HA, coords, RUBIN_LOCATION)
    ax.plot(HA, sinz, lw=2, label=f"{key} ($\\delta={dec:.1f}°$)")

ax.set_xlabel("Angle horaire (degrés)")
ax.set_ylabel("$\\sin z$")
ax.set_title("$\\sin z$ – méthode vectorielle")
secax0 = ax.secondary_xaxis("top", functions=(lambda x: x / 15.0, lambda x: x * 15.0))
secax0.set_xlabel("HA (heures)")
ax.legend(fontsize=8, loc="upper right")

# ── Formule directe ───────────────────────────────────────────────────────────
ax = axes[1]
for key, (ra, dec) in DEEP_FIELDS.items():
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    sinz = sinz_vs_HA(HA, coords, RUBIN_LOCATION)
    ax.plot(HA, sinz, lw=2, label=f"{key} ($\\delta={dec:.1f}°$)")

ax.set_xlabel("Angle horaire (degrés)")
ax.set_ylabel("$\\sin z$")
ax.set_title("$\\sin z$ – formule analytique")
secax1 = ax.secondary_xaxis("top", functions=(lambda x: x / 15.0, lambda x: x * 15.0))
secax1.set_xlabel("HA (heures)")
ax.legend(fontsize=8, loc="upper right")

fig.suptitle("Sinus de l'angle zénithal pour les DDF LSST", fontsize=13)
savefig("LSST-DDF-sinz")
plt.show()

---
## 9. Angle parallactique en fonction du temps (vérification astroplan)

Validation croisée entre notre formule et `astroplan.plot_parallactic`.

In [ ]:
# Champ de référence pour la validation
selected_field_name = "COSMOS"
coordinates = SkyCoord(
    DEEP_FIELDS[selected_field_name][0] * u.deg, DEEP_FIELDS[selected_field_name][1] * u.deg, frame="icrs"
)
field_target = FixedTarget(name=selected_field_name, coord=coordinates)

In [ ]:
# Grille temporelle sur une journée (pas = 1 heure)
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-01-01 23:59:59")
n_hours = int((t_end - t_start).to(u.hour).value)
times_day = t_start + np.arange(n_hours) * u.hour

q_day = calculate_parallactic_angle(coordinates, times_day, RUBIN_LOCATION)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")

# Notre calcul
axes[0].plot(times_day.to_datetime(), q_day, lw=2, color="steelblue")
axes[0].xaxis.set_major_locator(mdates.HourLocator(interval=3))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[0].xaxis.set_minor_locator(mdates.MinuteLocator(interval=30))
fig.autofmt_xdate()
axes[0].set_title("Notre calcul (COSMOS, 2026-01-01)")
axes[0].set_ylabel("$q$ (degrés)")

# Astroplan
plot_parallactic(field_target, observer, Time("2026-01-01T00:00:00"), ax=axes[1])
axes[1].xaxis.set_major_locator(mdates.HourLocator(interval=3))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[1].set_title("astroplan.plot_parallactic")

fig.suptitle("Validation croisée de l'angle parallactique (COSMOS)", fontsize=12)
plt.show()

---
## 10. Visibilité et masse d'air (COSMOS, jan–jun 2026)

In [ ]:
# Grille temporelle sur 6 mois (pas = 1 heure)
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-06-30 23:59:59")
n_hours = int((t_end - t_start).to(u.hour).value)
times_6m = t_start + np.arange(n_hours) * u.hour

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_airmass(field_target, observer, times_6m, ax=ax, brightness_shading=True)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.set_ylim(2.25, 1)
plt.xticks(rotation=45)
ax.set_title("Masse d'air – COSMOS (jan–jun 2026)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), dpi=120)
plot_airmass(
    field_target,
    observer,
    times_6m,
    ax=ax,
    brightness_shading=True,
    altitude_yaxis=True,
)

airmass_max = 2.2
ax.axhline(airmass_max, ls="--", lw=2, color="red", alpha=0.7)
ax.text(times_6m[0].datetime, airmass_max, f"Rubin limit (X={airmass_max:.1f})", color="red")

ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d"))
ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))
fig.autofmt_xdate()

ax.set_title("Visibilité COSMOS (jan–jun 2026)")
ax.set_ylabel("Masse d'air")
plt.tight_layout()
plt.show()

### 10.1 Carte heatmap de la masse d'air (COSMOS, nuit seule)

In [ ]:
# Coordonnées AltAz et masque nuit astronomique
altaz_frame = observer.altaz(times_6m, target=field_target)
airmass_arr = altaz_frame.secz
sun_alt = observer.altaz(times_6m, get_sun(times_6m)).alt
night_mask = sun_alt < -18 * u.deg

# Construction de la grille date × heure
local_times = times_6m.to_datetime(timezone=observer.timezone)
hours = np.array([t.hour + t.minute / 60.0 for t in local_times])
dates = np.array([t.date() for t in local_times])
unique_dates = np.unique(dates)
n_days = len(unique_dates)

grid = np.full((24, n_days), np.nan)
for i, (d, h, am, is_night) in enumerate(zip(dates, hours, airmass_arr, night_mask)):
    if is_night and am > 0:
        day_idx = np.where(unique_dates == d)[0][0]
        hour_idx = int(h)
        grid[hour_idx, day_idx] = float(am)

# Heatmap
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(grid, origin="lower", aspect="auto", vmin=1, vmax=2)
ax.set_xticks(np.arange(0, n_days, 14))
ax.set_xticklabels([str(d) for d in unique_dates[::14]], rotation=45)
ax.set_yticks(np.arange(0, 24, 2))
ax.set_ylabel("Heure locale")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Masse d'air")
ax.set_title("Heatmap de visibilité – COSMOS (nuit seule, jan–jun 2026)")
fig.tight_layout()
plt.show()

---
## 11. Vue polaire du ciel – position des DDF à un instant donné

In [ ]:
date_str = "2026-06-01 03:00:00"
observe_time = Time(date_str)

fig, ax = plt.subplots(1, 1, figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
ax.set_theta_direction(1)  # Est à gauche

for key, (ra, dec) in DEEP_FIELDS.items():
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    ft = FixedTarget(name=key, coord=coords)
    plot_sky(ft, observer, observe_time, ax, style_kwargs=DEEP_FIELDS_COLORSTYLE[key])

ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.05, 1))
ax.set_title(f"DDF LSST – {date_str} UTC")
plt.show()